# FLOW-IQ Day 1 — TRACE Flood Risk Pipeline

This notebook automates the **Day 1** workflow for flood risk assessment in Bwaise, Kampala.

### Workflow Steps:
1. **OpenTopography**: Download GLO-30 DEM.
2. **FABDEM Correction**: Strip buildings/trees for realistic urban flow analysis.
3. **WhiteboxTools**: D8 Flow Accumulation & Contour generation.
4. **Overpass Query**: Extract drains and roads from OSM.
5. **Synthetic Sampling**: 50 points for model validation (seed=42).
6. **XGBoost Gap Model**: Score urban gaps 0–1 for flood susceptibility.

### Deliverables (Google Drive):
1. `drains_osm.geojson`
2. `drains_cv.geojson` 
3. `flow_accumulation.geojson` 
4. `risk_zones.geojson` 
5. `bwaise_boundary.geojson` 

In [ ]:
# @title 1. Environment Setup
# Note: the package name for WhiteboxTools is 'whitebox'
!pip install whitebox geopandas osmnx xgboost rasterio shapely -q

import os
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import osmnx as ox
import xgboost as xgb
from shapely.geometry import shape, mapping, box, Point, MultiPolygon
from rasterio.features import shapes
from whitebox.whitebox_tools import WhiteboxTools
from google.colab import drive

# Mount Drive
drive.mount('/content/drive', force_remount=True)
OUTPUT_DIR = '/content/drive/MyDrive/FLOW-IQ/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

wbt = WhiteboxTools()
wbt.set_working_dir('.')

# Bwaise Bounding Box
WEST, SOUTH, EAST, NORTH = 32.55, 0.34, 32.57, 0.36
BBOX = [WEST, SOUTH, EAST, NORTH]
print("Setup Complete. Directory:", OUTPUT_DIR)

In [ ]:
# @title 2. OpenTopography GLO-30 Download
print("Downloading GLO-30 DEM from OpenTopography...")

url = f"https://portal.opentopography.org/API/globaldem?demtype=COP30&west={WEST}&south={SOUTH}&east={EAST}&north={NORTH}&outputFormat=GTiff"
response = requests.get(url)

if response.status_code == 200:
    with open("glo30_raw.tif", "wb") as f:
        f.write(response.content)
    print("✅ GLO-30 downloaded successfully.")
else:
    print("❌ Download failed. Status Code:", response.status_code)
    # Fallback to local file check if running locally
    if not os.path.exists("glo30_raw.tif"):
        raise Exception("DEM file missing and download failed.")

In [ ]:
# @title 3. FABDEM Building Correction
print("Running building/vegetation correction...")

# We use WhiteboxTools to remove off-terrain objects (buildings/trees)
# to create a bare-earth DTM suitable for hydrological analysis.
wbt.remove_off_terrain_objects("glo30_raw.tif", "conditioned_dem.tif", filter=11, slope=15.0)

print("✅ Correction complete: conditioned_dem.tif")

In [ ]:
# @title 4. D8 Flow Accumulation & Contours
print("Calculating flow accumulation...")

wbt.breach_depressions("conditioned_dem.tif", "breached.tif")
wbt.d8_flow_accumulation("breached.tif", "flow_acc.tif")

print("Generating GeoJSON contours...")
with rasterio.open("flow_acc.tif") as src:
    data = src.read(1)
    # Extract top 5% flow paths as contours
    threshold = np.percentile(data[data > 0], 95)
    mask = data > threshold
    
    results = (
        {'properties': {'flow_value': v}, 'geometry': shape(s)}
        for i, (s, v) in enumerate(shapes(data, mask=mask, transform=src.transform))
    )
    flow_gdf = gpd.GeoDataFrame.from_features(list(results), crs="EPSG:4326")
    flow_gdf.to_file(os.path.join(OUTPUT_DIR, "flow_accumulation.geojson"), driver='GeoJSON')

print("✅ Flow accumulation contours saved.")

In [ ]:
# @title 5. Infrastructure & Synthetic Sampling
print("Querying Overpass for drains...")
tags = {'waterway': 'drain'}
drains = ox.features_from_bbox(NORTH, SOUTH, EAST, WEST, tags=tags)
drains.to_file(os.path.join(OUTPUT_DIR, "drains_osm.geojson"), driver='GeoJSON')

print("Generating 50 synthetic CV points...")
np.random.seed(42)
lons = np.random.uniform(WEST, EAST, 50)
lats = np.random.uniform(SOUTH, NORTH, 50)
points = [Point(x, y) for x, y in zip(lons, lats)]
cv_gdf = gpd.GeoDataFrame(geometry=points, crs="EPSG:4326")
cv_gdf['id'] = range(50)
cv_gdf.to_file(os.path.join(OUTPUT_DIR, "drains_cv.geojson"), driver='GeoJSON')

# Create Bwaise Boundary
boundary = gpd.GeoDataFrame(geometry=[box(WEST, SOUTH, EAST, NORTH)], crs="EPSG:4326")
boundary.to_file(os.path.join(OUTPUT_DIR, "bwaise_boundary.geojson"), driver='GeoJSON')

print("✅ Infrastructure and sampling complete.")

In [ ]:
# @title 6. XGBoost Gap Scoring
print("Building risk scoring model...")

# 1. Generate Gap Polygons (Grid)
grid_size = 0.001 # approx 100m
x_coords = np.arange(WEST, EAST, grid_size)
y_coords = np.arange(SOUTH, NORTH, grid_size)
polygons = []
for x in x_coords:
    for y in y_coords:
        polygons.append(box(x, y, x + grid_size, y + grid_size))
gap_gdf = gpd.GeoDataFrame(geometry=polygons, crs="EPSG:4326")

# 2. Sample Features
def sample_at_center(raster_path, gdf):
    with rasterio.open(raster_path) as src:
        coords = [(p.centroid.x, p.centroid.y) for p in gdf.geometry]
        return [val[0] for val in src.sample(coords)]

gap_gdf['elevation'] = sample_at_center("conditioned_dem.tif", gap_gdf)
gap_gdf['flow_acc'] = sample_at_center("flow_acc.tif", gap_gdf)

# Distance to drain feature
drain_geom = drains.unary_union
gap_gdf['dist_drain'] = gap_gdf.geometry.centroid.apply(lambda p: p.distance(drain_geom))

# 3. Synthetic Training Labels for Demonstration
# In reality, use cv_gdf labels. Here we simulate risk.
X = gap_gdf[['elevation', 'flow_acc', 'dist_drain']]
y_sim = (X['flow_acc'] > X['flow_acc'].median()) & (X['elevation'] < X['elevation'].median())

model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, objective='reg:logistic')
model.fit(X, y_sim.astype(int))

# Score 0-1
gap_gdf['risk_score'] = model.predict(X)
gap_gdf.to_file(os.path.join(OUTPUT_DIR, "risk_zones.geojson"), driver='GeoJSON')

print("✅ XGBoost Gap Scoring complete.")

In [ ]:
# @title 7. Final Summary
print("--- FLOW-IQ DAY 1 COMPLETE ---")
print(f"Files saved to: {OUTPUT_DIR}")
for f in os.listdir(OUTPUT_DIR):
    if f.endswith('.geojson'):
        print(f"  📄 {f}")